# Nikolaisen2022 bin 00: smallest STL files

Evaluate surface quality, convert STL files to tetrahedral Gmsh `.msh` meshes with the Gmsh Delaunay 3D algorithm, print mesh quality and Merrill.jl memory estimates, and display a limited set of converted meshes with PyVista.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd()
if not (REPO / "src").exists() and (REPO.parent / "src").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from stl2fem.datasets import assign_size_bins, nikolaisen_inventory
from stl2fem.visualize import display_volume_mesh
from stl2fem.workflow import process_nikolaisen_size_bin

DATASET_ROOT = REPO / "data" / "Nikolaisen2022"
OUTPUT_ROOT = REPO / "processed" / "Nikolaisen2022"
BIN_INDEX = 0
N_BINS = 4
INPUT_UNIT = "um"
TARGET_EDGE_LENGTH_M = 9e-9
OVERWRITE = False
MAX_MESHES = None
DISPLAY_LIMIT = 8

In [ ]:
inventory = assign_size_bins(nikolaisen_inventory(DATASET_ROOT), n_bins=N_BINS)
subset = inventory[inventory["size_bin_index"] == BIN_INDEX]
print(f"Meshes in this notebook: {len(subset)}")
subset[["particle_id", "phase", "stl_format", "stl_size_mib", "size_bin", "source_path"]]

In [ ]:
results = process_nikolaisen_size_bin(
    DATASET_ROOT,
    OUTPUT_ROOT,
    bin_index=BIN_INDEX,
    n_bins=N_BINS,
    input_unit=INPUT_UNIT,
    target_edge_length_m=TARGET_EDGE_LENGTH_M,
    overwrite=OVERWRITE,
    max_meshes=MAX_MESHES,
)

columns = [
    "particle_id", "phase", "status", "stl_size_mib", "msh_size_bytes", "msh_size_mib",
    "input_unit", "input_scale_to_meters", "target_edge_length_m", "target_edge_length_native",
    "edge_length_median", "edge_length_p95", "edge_length_median_m", "edge_length_p95_m",
    "n_nodes", "n_tets", "volume_min", "scaled_jacobian_min",
    "radius_ratio_max", "estimated_memory_human", "msh_path", "merrill_msh_path", "merrill_msh_size_mib",
]
available = [column for column in columns if column in results.columns]
results[available]

In [ ]:
ok = results[results["status"] == "ok"].sort_values("msh_size_bytes")
display_count = len(ok) if DISPLAY_LIMIT is None else min(DISPLAY_LIMIT, len(ok))
print(f"Displaying {display_count} of {len(ok)} converted meshes. Increase DISPLAY_LIMIT to inspect more.")

for _, row in ok.head(display_count).iterrows():
    print(f"{row['particle_id']}: {row['n_nodes']} nodes, {row['n_tets']} tets, {row['estimated_memory_human']} estimated Merrill.jl memory")
    print(f"  native: {row['msh_path']}")
    print(f"  Merrill meters: {row.get('merrill_msh_path', '')}")
    display_volume_mesh(row["msh_path"], show_edges=True, opacity=0.35)